In [ ]:
!pip install datasets transformers accelerate pyvi

In [ ]:
model_checkpoint = "vinai/phobert-base"
batch_size = 16

In [ ]:
from datasets import load_dataset, ClassLabel

# 1. Định nghĩa nhãn chuẩn
label_list = ["negative", "neutral", "positive"]
class_names = ClassLabel(names=label_list)

# 2. Load dataset từ 2 file riêng biệt
# Cấu hình data_files dạng dictionary tự động tạo ra các split 'train' và 'eval' tương ứng
raw_datasets = load_dataset("json", data_files={
    "train": "vi_train.json",
    "eval": "vi_test.json"
})

# 3. Hàm chuẩn hóa và lọc dữ liệu
def clean_and_validate(example):
    sentiment = example.get("sentiment")
    if not isinstance(sentiment, str):
        return False
    sentiment = sentiment.strip().lower()
    return sentiment in label_list

# Áp dụng bộ lọc cho cả 2 tập dữ liệu trong DatasetDict
dataset_clean = raw_datasets.filter(clean_and_validate)

# 4. Map label và Cast column
def map_labels(example):
    sentiment = example["sentiment"].strip().lower()
    example["label"] = class_names.str2int(sentiment)
    return example

# Thực hiện map và cast trực tiếp trên DatasetDict để xử lý đồng thời cả 2 tập
dataset_mapped = dataset_clean.map(map_labels)
dataset_dict = dataset_mapped.cast_column("label", class_names)

print("--- Phân phối nhãn tập TRAIN ---")
print(dataset_dict["train"].to_pandas()["label"].value_counts().sort_index())

print("\n--- Phân phối nhãn tập EVAL (Từ file test riêng biệt) ---")
print(dataset_dict["eval"].to_pandas()["label"].value_counts().sort_index())

In [ ]:
label_list = ["negative", "neutral", "positive"]

def encode_label(example):
    example["label"] = label_list.index(example["sentiment"])
    return example

dataset = dataset_dict.map(encode_label)
dataset = dataset.cast_column("label", ClassLabel(names=label_list))

In [ ]:
import datasets
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [ ]:
show_random_elements(dataset["train"])

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=False)

In [ ]:
from pyvi import ViTokenizer

def preprocess_function(examples):
    texts = [ViTokenizer.tokenize(x) for x in examples["sentence"]]
    return tokenizer(texts, truncation=True, max_length=256)

In [ ]:
preprocess_function(dataset['train'][:5])

In [ ]:
encoded_dataset = dataset.map(preprocess_function, batched=True)

In [ ]:
print(dataset["train"].features["label"])

In [ ]:
from transformers import AutoModelForSequenceClassification

# Lấy label trực tiếp từ dataset (CHUẨN NHẤT)
label_list = dataset["train"].features["label"].names

# Tạo mapping
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Debug (nên có)
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)

In [ ]:
from transformers import TrainingArguments

In [ ]:
# 1. Chọn F1 làm thước đo chính để chọn model tốt nhất (vì dữ liệu tài chính hay lệch)
metric_name = "f1"
model_name = model_checkpoint.split("/")[-1]

args = TrainingArguments(
    f"{model_name}-finetuned-vfa-sentiment-optimized",
    eval_strategy="epoch",
    save_strategy="epoch",

    # 1. Tinh chỉnh tốc độ học (Thấp hơn một chút để học kỹ hơn)
    learning_rate=1.5e-5,

    # 2. Tăng khả năng chống học vẹt (Regularization)
    weight_decay=0.1,            # Tăng từ 0.01 lên 0.1 để kiểm soát trọng số chặt hơn
    label_smoothing_factor=0.1,  # CỰC KỲ QUAN TRỌNG: Giúp model không quá tự tin vào nhãn giả

    # 3. Ổn định hóa quá trình học ở giai đoạn đầu và cuối
    warmup_ratio=0.1,            # Dành 10% thời gian đầu để "khởi động" nhẹ nhàng
    lr_scheduler_type="cosine",  # Giảm tốc độ học theo đường cong Cosine (mượt hơn mặc định)

    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=7,          # Tăng lên 7 epoch vì chúng ta có Early Stopping và Load Best

    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    greater_is_better=True,
    logging_steps=10,

    # 4. Tiết kiệm bộ nhớ và hỗ trợ tập trung (Tùy chọn)
    group_by_length=True,        # Gom các câu cùng chiều dài để train nhanh và ổn định hơn
)

In [ ]:
from transformers import Trainer

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    # 1. Accuracy (Độ chính xác tổng thể)
    acc = accuracy_score(labels, predictions)

    # 2. Weighted F1 (Thước đo chính bạn đang dùng)
    weighted_f1 = f1_score(labels, predictions, average="weighted")

    # 3. Macro F1 (CỰC KỲ QUAN TRỌNG)
    # Nó tính trung bình F1 của 3 nhãn mà không quan tâm nhãn đó nhiều hay ít data.
    # Nếu Macro F1 thấp hơn Weighted F1 nhiều -> Model đang học kém ở các nhãn ít data (như Positive hoặc Negative).
    macro_f1 = f1_score(labels, predictions, average="macro")

    # 4. Precision và Recall (Để biết model có đang "đoán bừa" không)
    precision = precision_score(labels, predictions, average="weighted")
    recall = recall_score(labels, predictions, average="weighted")

    return {
        "accuracy": acc,
        "f1": weighted_f1,      # Trainer sẽ dùng key này để chọn model tốt nhất
        "macro_f1": macro_f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
from transformers import DataCollatorWithPadding, EarlyStoppingCallback

# 1. Tạo data collator (giúp padding linh hoạt theo từng batch)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 2. Khởi tạo Trainer với cơ chế Dừng sớm (Early Stopping)
trainer = Trainer(
    model=model,
    args=args, # Đảm bảo bạn đang dùng bộ args có label_smoothing và weight_decay đã bàn ở trên
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["eval"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,

    # THÊM DÒNG NÀY:
    # Nếu sau 2 lần Eval (2 epoch) mà F1 không tăng thì dừng ngay để tránh học vẹt
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
# --- CHẠY BENCHMARK TRÊN FILE TEST_DATASET.JSON CỐ ĐỊNH ---

# 1. Load file benchmark riêng
test_raw = load_dataset("json", data_files={"test": "/content/vi_test.json"})

# 2. Tiền xử lý tập test (Lọc -> Map nhãn -> Cast)
test_clean = test_raw["test"].filter(clean_and_validate)
test_mapped = test_clean.map(map_labels)
test_final = test_mapped.cast_column("label", class_names)

# 3. Tokenize (Sử dụng preprocess_function có ViTokenizer đã khai báo ở trên)
encoded_test = test_final.map(preprocess_function, batched=True)

# 4. Chạy dự báo
print("\n" + "="*50)
print("KẾT QUẢ BENCHMARK TRÊN TẬP TEST ĐỘC LẬP")
print("="*50)

# trainer.predict sẽ tự dùng bộ tham số tốt nhất (nếu bạn đã set load_best_model_at_end=True)
results = trainer.predict(encoded_test)

# 5. In kết quả các chỉ số
metrics = results.metrics
print(f"Số lượng mẫu: {len(encoded_test)}")
print(f"Accuracy: {metrics['test_accuracy']:.4f}")
print(f"F1 (Weighted): {metrics['test_f1']:.4f}")
print(f"F1 (Macro): {metrics['test_macro_f1']:.4f}")
print(f"Precision: {metrics['test_precision']:.4f}")
print(f"Recall: {metrics['test_recall']:.4f}")
print("="*50)

# 6. Lưu kết quả dự đoán sai ra file để soi lỗi
import pandas as pd
import numpy as np

preds = np.argmax(results.predictions, axis=1)
labels = results.label_ids

df_errors = pd.DataFrame({
    "sentence": test_final["sentence"],
    "true_label": [label_list[i] for i in labels],
    "pred_label": [label_list[i] for i in preds]
})
df_errors = df_errors[df_errors["true_label"] != df_errors["pred_label"]]
df_errors.to_csv("final_benchmark_errors.csv", index=False, encoding='utf-8-sig')
print(f"Đã lưu {len(df_errors)} dòng dự đoán sai vào file final_benchmark_errors.csv")